## Arbol de decision

In [ ]:
# ==========================================
# NOTEBOOK: 01_arbol_decision.ipynb
# OBJETIVO:
# Predecir si una mujer abandona o no su carrera tech
# usando un Árbol de Decisión
# ==========================================

# ==========
# 1. LIBRERÍAS
# ==========
import os
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# ==========
# 2. DETECTAR RUTA BASE
# ==========
print("Carpeta actual:", os.getcwd())

if Path("data/processed").exists():
    BASE_PATH = Path(".")
elif Path("../data/processed").exists():
    BASE_PATH = Path("..")
else:
    raise FileNotFoundError("No se encontró la carpeta data/processed")

PROCESSED_PATH = BASE_PATH / "data" / "processed"

# ==========
# 3. CARGAR DATASET LIMPIO
# ==========
file_path = PROCESSED_PATH / "women_tech_attrition_clean.csv"
df = pd.read_csv(file_path)

print("\nShape del dataset:", df.shape)
display(df.head())

# ==========
# 4. REVISIÓN BÁSICA
# ==========
print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

# ==========
# 5. PREPARAR VARIABLE OBJETIVO
# Convertimos abandona: si/no -> 1/0
# ==========
df["abandona"] = df["abandona"].map({"si": 1, "no": 0})

print("\nDistribución de la variable objetivo:")
print(df["abandona"].value_counts())

# ==========
# 6. SELECCIONAR VARIABLES
# Usamos variables simples y fáciles de explicar
# ==========
features = [
    "edad",
    "anios_experiencia",
    "salario_anual",
    "tamano_empresa",
    "teletrabajo",
    "politicas_conciliacion",
    "satisfaccion_laboral",
    "brecha_salarial_percibida",
    "anios_en_puesto"
]

target = "abandona"

X = df[features].copy()
y = df[target].copy()

# ==========
# 7. CODIFICAR COLUMNAS CATEGÓRICAS
# El árbol necesita números, así que convertimos texto a números
# ==========
label_encoders = {}

for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

# ==========
# 8. DIVIDIR EN TRAIN Y TEST
# ==========
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTamaño train:", X_train.shape)
print("Tamaño test:", X_test.shape)

# ==========
# 9. ENTRENAR EL MODELO
# ==========
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

# ==========
# 10. HACER PREDICCIONES
# ==========
y_pred = model.predict(X_test)

# ==========
# 11. EVALUAR EL MODELO
# ==========
accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy del modelo:", round(accuracy, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusión:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# ==========
# 12. VISUALIZAR MATRIZ DE CONFUSIÓN
# ==========
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusión - Árbol de Decisión")
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.show()

# ==========
# 13. IMPORTANCIA DE VARIABLES
# ==========
importance_df = pd.DataFrame({
    "variable": X.columns,
    "importancia": model.feature_importances_
}).sort_values(by="importancia", ascending=False)

print("\nImportancia de variables:")
display(importance_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=importance_df, x="importancia", y="variable", hue="variable", dodge=False, legend=False)
plt.title("Importancia de variables")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.show()

# ==========
# 14. VISUALIZAR EL ÁRBOL
# ==========
plt.figure(figsize=(20, 10))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=["no abandona", "abandona"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árbol de Decisión - Abandono de carrera")
plt.show()

# ==========
# 15. EJEMPLO DE PREDICCIÓN
# ==========
sample = X_test.iloc[[0]]
sample_pred = model.predict(sample)[0]

print("\nEjemplo de predicción:")
display(sample)

if sample_pred == 1:
    print("Predicción: abandona")
else:
    print("Predicción: no abandona")
